## ANALISIS CROSS SELLING PARA CAMPAÑAS
### CÁLCULO DE CORRELACION ENTRE PARES SKUs 
Autor: Flavia Davila   
2026

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter, defaultdict

## ANALISIS COMPARATIVO

### PERIODO ANTERIOR (LY)

#### CARGA DE DATOS

In [ ]:
# Leemos la hoja armada en base al cubo para extraer el arbol de categorizacion
cubo = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\cubo_1.xlsx", 
                    sheet_name = 'SKUs',
                    skiprows=4
                    )

In [ ]:
# leemos el csv completo de ventas facturadas del periodo 
ventasune_farma = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\scripts\febrero_2026.csv")

In [ ]:
# leemos informacion de los clusters de la UNE
clusters = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\clusters_bdgs.xlsx")

#### TRANSFORMACION PRELIMINAR

In [ ]:
#convertimos fechas a formato datetime en caso de querer filtrar periodos especificos
ventasune_farma['FECHA_FACT']= pd.to_datetime(ventasune_farma['FECHA_FACT'])

#### FILTRO DE DATA DE INTERÉS

In [ ]:
#analizamos el periodo de interés
#ventasxfact_farma = ventasune_farma[(ventasune_farma['FECHA_FACT'] >= '2025-03-01 00:00:00') & (ventasune_farma['FECHA_FACT'] <= '2025-03-15 00:00:00')]
ventasxfact_farma = ventasune_farma.copy()

#### TRANSFORMACION DE LA DATA

In [ ]:
# Agrupamos para contar cantidad de facturas emitidas por dia en el periodo analizado
ventasfactxdia = (
    ventasxfact_farma
    .groupby('FECHA_FACT')['NUMERO_FACTURA']
    .nunique()
    .reset_index(name='FACTURAS_UNICAS')
)


In [ ]:
# guardamos la cantidad de facturas unicas emitidas en el periodo analizado
total_fact_periodo = ventasfactxdia['FACTURAS_UNICAS'].sum()
total_fact_periodo

In [ ]:
# filtramos solo BODEGAS de la UNE de interés
#clusters_amkt = clusters[clusters['UNE']=='AMARKET']
clusters_farma = clusters[clusters['UNE']=='FARMACORP']

### Procesamos la data

In [ ]:
# convertimos los IDs en string
cubo['COD_ARTICULO'] = cubo['COD_ARTICULO'].astype(str)
ventasxfact_farma['COD_ARTICULO'] = ventasxfact_farma['COD_ARTICULO'].astype(str)

In [ ]:
# unimos la informacion de los SKUs con el conteo de ventasxfactura
ventas_farma = pd.merge(cubo,
                        ventasxfact_farma,
                        on='COD_ARTICULO',
                        how='inner'
                        )

In [ ]:
# traemos la info de los clusters 
ventas_farmacorp = pd.merge(
    ventas_farma, 
    clusters_farma,
    left_on='COD_BODEGA',
    right_on ='BODEGA',
    how='left')

### Apertura por REGION

In [ ]:
ventas_farmacorp.CIUDAD.value_counts()

In [ ]:
# Filtramos por ciudad ('COCHABAMBA', 'LA PAZ', 'TARIJA', 'BENI', 'ORURO', 'PANDO', 'SUCRE', 'POTOSI')
#ciudad = 'SANTA CRUZ' 
#ventas_region_farma = ventas_farmacorp[ventas_farmacorp['CIUDAD']==ciudad]

# Todas las ventas
ventas_region_farma = ventas_farmacorp.copy()

In [ ]:
# Calculamos pares de SKUs frecuencia de ocurrencia conjunta y moda de ocurrencias.
pares = Counter()
unidades_pares = defaultdict(Counter)

cat1_map = ventas_region_farma.set_index('ARTICULO')['CAT 2'].to_dict()

ventas_sorted = ventas_region_farma[['NUMERO_FACTURA', 'ARTICULO', 'UNIDADES']].sort_values('NUMERO_FACTURA')

factura_actual = None
articulos_actuales = []

for row in ventas_sorted.itertuples(index=False):
    if row.NUMERO_FACTURA != factura_actual:
        if len(articulos_actuales) >= 2:

            articulos_ordenados = sorted(articulos_actuales, key=lambda x: x[0])
            for (art_a, und_a), (art_b, und_b) in combinations(articulos_ordenados, 2):
                if cat1_map.get(art_a) != cat1_map.get(art_b):

                    pares[(art_a, art_b)] += 1
                    unidades_pares[(art_a, art_b)][(und_a, und_b)] += 1
                    
        factura_actual = row.NUMERO_FACTURA
        articulos_actuales = [(row.ARTICULO, row.UNIDADES)]
    else:
        articulos_actuales.append((row.ARTICULO, row.UNIDADES))

if len(articulos_actuales) >= 2:
    articulos_ordenados = sorted(articulos_actuales, key=lambda x: x[0])
    for (art_a, und_a), (art_b, und_b) in combinations(articulos_ordenados, 2):
        if cat1_map.get(art_a) != cat1_map.get(art_b):
            pares[(art_a, art_b)] += 1
            unidades_pares[(art_a, art_b)][(und_a, und_b)] += 1


resultados_pares = []
for (a, b), freq in pares.items():
    (moda_und_a, moda_und_b), frec_moda = unidades_pares[(a, b)].most_common(1)[0]
    
    resultados_pares.append({
        'ARTICULO_A': a,
        'ARTICULO_B': b,
        'FRECUENCIA': freq,
        'MODA_UNIDADES_A': moda_und_a,
        'MODA_UNIDADES_B': moda_und_b,
        'FREC_MODA_UNIDADES': frec_moda
    })

df_pares = pd.DataFrame(resultados_pares)

cat_cols = ['ARTICULO', 'CAT 1', 'CAT 2', 'CAT 3', 'CAT 4']
attrs = ventas_region_farma.drop_duplicates('ARTICULO').set_index('ARTICULO')[cat_cols[1:]]

df_pares = df_pares.join(attrs.add_suffix('_A'), on='ARTICULO_A')
df_pares = df_pares.join(attrs.add_suffix('_B'), on='ARTICULO_B')

df_pares = df_pares.sort_values('FRECUENCIA', ascending=False).reset_index(drop=True)

In [ ]:
# Calculamos cuántas facturas únicas tiene cada artículo por separado
frecuencias_individuales = ventas_region_farma.groupby('ARTICULO')['NUMERO_FACTURA'].nunique().to_dict()

# Mapeamos esas frecuencias a los artículos A y B en nuestro DataFrame de pares
df_pares['FREQ_A'] = df_pares['ARTICULO_A'].map(frecuencias_individuales)
df_pares['FREQ_B'] = df_pares['ARTICULO_B'].map(frecuencias_individuales)

# Calculamos la correlación (Confianza) para A y para B
df_pares['CONF_A'] = df_pares['FRECUENCIA'] / df_pares['FREQ_A']
df_pares['CONF_B'] = df_pares['FRECUENCIA'] / df_pares['FREQ_B']


In [ ]:
# copiamos el DF
pairs_farma_region = df_pares.copy()

In [ ]:
# Calculamos la tasa de frecuencia en relacion al total de facturas emitidas
pairs_farma_region['TASA_AB'] = pairs_farma_region['FRECUENCIA']/ total_fact_periodo

In [ ]:
pairs_2025 = pairs_farma_region.copy()

In [ ]:
# Filtramos pares de SKUs con una frecuencia mayor a 10 en conjunto
final_pairs = pairs_2025[pairs_2025['FRECUENCIA']>=10]

In [ ]:
final_pairs.shape

In [ ]:
# Exportamos a un excel
output = 'feb_2026' # <- cambiamos el nombre a requerimiento

final_pairs.to_excel(f"C:\\Users\\fdavila\\OneDrive - Farmacorp S.A\\Escritorio\\cross selling\\output\\{output}.xlsx")

### PERIODO ACTUAL

In [ ]:
ventasune_farma_2026 = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\scripts\resultado.csv")

In [ ]:
ventasune_farma_2026['NUMERO_FACTURA'].nunique()

In [ ]:
#convertimos fechas a formato datetime
ventasune_farma_2026['FECHA_FACT']= pd.to_datetime(ventasune_farma_2026['FECHA_FACT'])

In [ ]:
#analizamos los dias de carnavales
ventasxfact_farma_26 = ventasune_farma_2026[(ventasune_farma_2026['FECHA_FACT'] >= '2026-02-13 00:00:00') & (ventasune_farma_2026['FECHA_FACT'] <= '2026-02-28 00:00:00')]

In [ ]:
ventasfactxdia = (
    ventasxfact_farma_26
    .groupby('FECHA_FACT')['NUMERO_FACTURA']
    .nunique()
    .reset_index(name='FACTURAS_UNICAS')
)


In [ ]:
ventasfactxdia.shape

In [ ]:
total_fact_periodo_26 = ventasfactxdia['FACTURAS_UNICAS'].sum()
total_fact_periodo_26

In [ ]:
ventasfactxdia['FACTURAS_UNICAS'][ventasfactxdia['FECHA_FACT'].between('2026-02-13', '2026-02-28')].sum()


In [ ]:
cubo['COD_ARTICULO'] = cubo['COD_ARTICULO'].astype(str)
ventasxfact_farma_26['COD_ARTICULO'] = ventasxfact_farma_26['COD_ARTICULO'].astype(str)

In [ ]:
# filtering only SKUs on FARMACORP
ventas_farma = pd.merge(cubo,
                        ventasxfact_farma_26,
                        on='COD_ARTICULO',
                        how='inner'
                        )

In [ ]:
ventas_farmacorp = pd.merge(
    ventas_farma, 
    clusters_farma,
    left_on='COD_BODEGA',
    right_on ='BODEGA',
    how='left')

In [ ]:
#ventas_region_farma = ventas_farmacorp[ventas_farmacorp['CIUDAD']=='SANTA CRUZ']
ventas_region_farma = ventas_farmacorp.copy()

In [ ]:
pares = Counter()
unidades_pares = defaultdict(Counter)

# Traer CAT 1 como dict para lookup O(1)
cat1_map = ventas_region_farma.set_index('ARTICULO')['CAT 2'].to_dict()

# AGREGADO: Incluir la columna 'UNIDADES' en la selección
ventas_sorted = ventas_region_farma[['NUMERO_FACTURA', 'ARTICULO', 'UNIDADES']].sort_values('NUMERO_FACTURA')

factura_actual = None
articulos_actuales = []

for row in ventas_sorted.itertuples(index=False):
    if row.NUMERO_FACTURA != factura_actual:
        if len(articulos_actuales) >= 2:
            articulos_ordenados = sorted(articulos_actuales, key=lambda x: x[0])
            for (art_a, und_a), (art_b, und_b) in combinations(articulos_ordenados, 2):
                if cat1_map.get(art_a) != cat1_map.get(art_b):
                    pares[(art_a, art_b)] += 1
                    unidades_pares[(art_a, art_b)][(und_a, und_b)] += 1
                    
        factura_actual = row.NUMERO_FACTURA
        articulos_actuales = [(row.ARTICULO, row.UNIDADES)]
    else:
        articulos_actuales.append((row.ARTICULO, row.UNIDADES))

# Último grupo
if cat1_map.get(art_a) != cat1_map.get(art_b):
            pares[(art_a, art_b)] += 1
            unidades_pares[(art_a, art_b)][(und_a, und_b)] += 1
# Armar el DataFrame extrayendo la moda
resultados_pares = []
for (a, b), freq in pares.items():
    (moda_und_a, moda_und_b), frec_moda = unidades_pares[(a, b)].most_common(1)[0]
    
    resultados_pares.append({
        'ARTICULO_A': a,
        'ARTICULO_B': b,
        'FRECUENCIA': freq,
        'MODA_UNIDADES_A': moda_und_a,
        'MODA_UNIDADES_B': moda_und_b,
        'FREC_MODA_UNIDADES': frec_moda
    })
df_pares = pd.DataFrame(resultados_pares)
# Join con árbol de categorías completo
cat_cols = ['ARTICULO', 'CAT 1', 'CAT 2', 'CAT 3', 'CAT 4']
attrs = ventas_region_farma.drop_duplicates('ARTICULO').set_index('ARTICULO')[cat_cols[1:]]
df_pares = df_pares.join(attrs.add_suffix('_A'), on='ARTICULO_A')
df_pares = df_pares.join(attrs.add_suffix('_B'), on='ARTICULO_B')
df_pares = df_pares.sort_values('FRECUENCIA', ascending=False).reset_index(drop=True)

In [ ]:
# 1. Calculamos cuántas facturas únicas tiene cada artículo por separado
frecuencias_individuales = ventas_region_farma.groupby('ARTICULO')['NUMERO_FACTURA'].nunique().to_dict()

# 2. Mapeamos esas frecuencias a los artículos A y B en nuestro DataFrame de pares
df_pares['FREQ_A'] = df_pares['ARTICULO_A'].map(frecuencias_individuales)
df_pares['FREQ_B'] = df_pares['ARTICULO_B'].map(frecuencias_individuales)

# 3. Calculamos la correlación (Confianza) para A y para B
df_pares['CONF_A'] = df_pares['FRECUENCIA'] / df_pares['FREQ_A']
df_pares['CONF_B'] = df_pares['FRECUENCIA'] / df_pares['FREQ_B']


In [ ]:
pairs_farma_region = df_pares.copy()

In [ ]:
pairs_farma_region['TASA_AB'] = pairs_farma_region['FRECUENCIA']/ total_fact_periodo_26

In [ ]:
pairs_2026 = pairs_farma_region.copy()

In [ ]:
pairs_2025.TASA_AB.describe()

In [ ]:
# Merge de ambos períodos
df_comparacion = pairs_2025.merge(
    pairs_2026,
    on=['ARTICULO_A', 'ARTICULO_B'],
    suffixes=('_SIN', '_CON'),
    how='outer'
).fillna(0)


In [ ]:
df_comparacion.sort_values('TASA_AB_CON', ascending=False).head(20)

In [ ]:
df_comparacion[['FRECUENCIA_SIN','TASA_AB_SIN', 'FRECUENCIA_CON', 'TASA_AB_CON']].sort_values('TASA_AB_CON', ascending=False).head(10)

In [ ]:
# Incremento absoluto (útil como segundo filtro)
df_comparacion['DELTA_TASA'] = df_comparacion['TASA_AB_CON'] - df_comparacion['TASA_AB_SIN']

In [ ]:
# 1. Pares POTENCIADOS por la promo (existían antes y crecieron)
potenciados = df_comparacion[
    (df_comparacion['TASA_AB_SIN'] > 0) &  # ajusta el umbral
    (df_comparacion['TASA_AB_CON'] >= df_comparacion['TASA_AB_SIN'])
    #(df_comparacion['TASA_AB_CON'] >= 0.0001)
]

# 2. Pares NUEVOS generados por la promo (no existían antes)
nuevos = df_comparacion[
    (df_comparacion['TASA_AB_SIN'] == 0) &
    (df_comparacion['TASA_AB_CON'] > 0.001)    # ajusta el umbral
]

# 3. Pares que CAYERON (la promo no ayudó o canibalizó)
caidos = df_comparacion[
    (df_comparacion['TASA_AB_SIN'] > 0) &
    (df_comparacion['DELTA_TASA'] < -0.001)    # ajusta el umbral
]

In [ ]:
potenciados.shape

In [ ]:
potenciados[potenciados.ARTICULO_B.isin(['RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA'])]

In [ ]:
potenciados.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\correlacion_AB_completo.xlsx")